# Python-native placement pairwise LOGO experiment

## 1. 실험 목표와 가설

이 노트북의 단일 목표는 Human placement 선택을 모방하는 고정 baseline을 Python만으로 재구성하는 것이다.

가설:

- 현재 16개 feature와 decision-balanced pairwise logistic regression은 game 단위 held-out 선택 순위를 일정 수준 설명한다.
- canonical Top-1과 tie-aware Top-1의 차이는 현재 feature 표현의 중복 한계를 드러낸다.
- 이 결과는 gameplay 강도나 승률이 아니라 Human 선택 모방 성능이다.

각 단계는 이유, 실행 코드, 계약 assertion, 작은 요약 output 순서로 구성한다.

In [ ]:
EXPERIMENT_NAME = "python-native-placement-pairwise-logo"
EXPERIMENT_HYPOTHESIS = (
    "The fixed 16-feature pairwise baseline reproduces a deterministic Human-choice "
    "imitation LOGO evaluation without JavaScript experiment dependencies."
)

# Assertion
assert EXPERIMENT_NAME and "Human-choice" in EXPERIMENT_HYPOTHESIS

# Output
{"experiment": EXPERIMENT_NAME, "hypothesis": EXPERIMENT_HYPOTHESIS}

## 2. 환경 및 라이브러리 버전

수치 결과의 재현 조건을 먼저 고정한다. 노트북은 environment-ml.yml에 명시된 버전과 다르면 즉시 중단한다.

In [ ]:
import hashlib
import io
import json
import math
import os
import platform
import tempfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import sklearn
from IPython.display import display
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

EXPECTED_ENVIRONMENT = {
    "python": "3.12.13",
    "numpy": "2.5.1",
    "pandas": "3.0.3",
    "scipy": "1.18.0",
    "scikitLearn": "1.9.0",
}
ACTUAL_ENVIRONMENT = {
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scipy": scipy.__version__,
    "scikitLearn": sklearn.__version__,
}

# Assertion
assert ACTUAL_ENVIRONMENT == EXPECTED_ENVIRONMENT, ACTUAL_ENVIRONMENT

# Output
display(pd.DataFrame([ACTUAL_ENVIRONMENT]))

## 3. 입력 경로와 source SHA-256

저장소 루트를 실행 기준으로 삼고 사용자별 절대 경로는 저장하지 않는다. 네 source 파일의 checksum을 학습 전에 기록하며 report 경로가 lexical path, realpath, symlink 또는 기존 inode로 source와 충돌하지 않는지 확인한다.

In [ ]:
def find_repository_root(start):
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "package.json").is_file() and (
            candidate / "scripts/export-placement-dataset.js"
        ).is_file():
            return candidate
    raise AssertionError("REPOSITORY_ROOT_NOT_FOUND")

ROOT = find_repository_root(Path.cwd())
DATASET_RELATIVE_DIR = Path(os.environ.get(
    "PLACEMENT_EXPERIMENT_DATASET",
    "data/datasets/human-behavior-v2-snapshot-26g-20260717",
))
assert not DATASET_RELATIVE_DIR.is_absolute(), "DATASET_PATH_MUST_BE_RELATIVE"
assert ".." not in DATASET_RELATIVE_DIR.parts, "DATASET_PATH_PARENT_ALIAS_NOT_ALLOWED"
DATASET_DIR = ROOT / DATASET_RELATIVE_DIR
SOURCE_FILES = {
    "manifest.json": DATASET_DIR / "manifest.json",
    "train.jsonl": DATASET_DIR / "train.jsonl",
    "validation.jsonl": DATASET_DIR / "validation.jsonl",
    "rejected.jsonl": DATASET_DIR / "rejected.jsonl",
}
report_value = os.environ.get(
    "PLACEMENT_EXPERIMENT_REPORT",
    "/private/tmp/placement-pairwise-logo-experiment-report.json",
)
REPORT_PATH = Path(report_value)
if not REPORT_PATH.is_absolute():
    REPORT_PATH = ROOT / REPORT_PATH

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def lexical_path(path):
    return Path(os.path.abspath(os.fspath(path)))

def same_inode(left, right):
    left_stat = left.stat()
    right_stat = right.stat()
    return (left_stat.st_dev, left_stat.st_ino) == (right_stat.st_dev, right_stat.st_ino)

def assert_output_separate_from_sources(output_path, source_paths):
    output_lexical = lexical_path(output_path)
    output_real = output_path.resolve(strict=False)
    if os.path.lexists(output_path) and output_path.is_symlink():
        raise AssertionError("REPORT_OUTPUT_SYMLINK_NOT_ALLOWED")
    for source_path in source_paths:
        if output_lexical == lexical_path(source_path):
            raise AssertionError("REPORT_OUTPUT_LEXICAL_SOURCE_COLLISION")
        if output_real == source_path.resolve(strict=True):
            raise AssertionError("REPORT_OUTPUT_REALPATH_SOURCE_COLLISION")
        if os.path.lexists(output_path) and same_inode(output_path, source_path):
            raise AssertionError("REPORT_OUTPUT_INODE_SOURCE_COLLISION")

# Assertion
assert (ROOT / "package.json").is_file()
assert all(path.is_file() for path in SOURCE_FILES.values())
assert_output_separate_from_sources(REPORT_PATH, list(SOURCE_FILES.values()))
SOURCE_SHA256_BEFORE = {name: sha256_file(path) for name, path in SOURCE_FILES.items()}
assert all(len(value) == 64 for value in SOURCE_SHA256_BEFORE.values())

# Output
display(pd.DataFrame(
    [{"source": name, "sha256": digest} for name, digest in SOURCE_SHA256_BEFORE.items()]
))

## 4. manifest/train/validation/rejected 로딩

UTF-8 strict decode와 행별 JSON object 검증을 먼저 수행한 뒤 pandas.read_json으로 JSONL을 직접 읽는다. 큰 원본 DataFrame 전체 대신 shape와 식별자 일부만 표시한다.

In [ ]:
def read_json_object(path):
    text = path.read_bytes().decode("utf-8", errors="strict")
    value = json.loads(text)
    assert isinstance(value, dict), f"JSON_OBJECT_REQUIRED:{path.name}"
    return value

def read_jsonl_frame(path):
    text = path.read_bytes().decode("utf-8", errors="strict")
    lines = text.splitlines()
    assert lines, f"EMPTY_JSONL:{path.name}"
    for line_number, line in enumerate(lines, start=1):
        assert line.strip(), f"EMPTY_JSONL_LINE:{path.name}:{line_number}"
        value = json.loads(line)
        assert isinstance(value, dict), f"JSON_OBJECT_LINE_REQUIRED:{path.name}:{line_number}"
    frame = pd.read_json(io.StringIO(text), lines=True, dtype=False)
    assert len(frame) == len(lines), f"PANDAS_ROW_COUNT_MISMATCH:{path.name}"
    return frame

manifest = read_json_object(SOURCE_FILES["manifest.json"])
train_source = read_jsonl_frame(SOURCE_FILES["train.jsonl"])
validation_source = read_jsonl_frame(SOURCE_FILES["validation.jsonl"])
rejected_source = read_jsonl_frame(SOURCE_FILES["rejected.jsonl"])

# Assertion
assert not train_source.empty and not validation_source.empty and not rejected_source.empty

# Output
display(pd.DataFrame([
    {"file": "train.jsonl", "rows": len(train_source), "columns": len(train_source.columns)},
    {"file": "validation.jsonl", "rows": len(validation_source), "columns": len(validation_source.columns)},
    {"file": "rejected.jsonl", "rows": len(rejected_source), "columns": len(rejected_source.columns)},
]))
display(train_source[["gameId", "playerId", "pieceIndex", "source", "selectedBySource"]].head(3))

## 5. source/schema/count 검증

LOGO 전에는 기존 train/validation split의 gameId가 겹치지 않는지 확인한 뒤 두 candidate set을 결합한다. source, dataset schema, boolean label과 manifest의 파일 단위 count를 검증한다.

In [ ]:
REQUIRED_CANDIDATE_COLUMNS = {
    "datasetSchemaVersion",
    "gameId",
    "playerId",
    "pieceIndex",
    "source",
    "aiDifficulty",
    "pieceType",
    "boardBefore",
    "candidatePlacement",
    "metricsBefore",
    "metricDelta",
    "candidateResult",
    "selectedBySource",
    "metadata",
}

def validate_candidate_source(frame, split_name):
    missing = REQUIRED_CANDIDATE_COLUMNS.difference(frame.columns)
    assert not missing, f"MISSING_COLUMNS:{split_name}:{sorted(missing)}"
    assert frame["datasetSchemaVersion"].map(
        lambda value: isinstance(value, (int, np.integer)) and not isinstance(value, bool)
        and int(value) == 2
    ).all(), f"DATASET_SCHEMA_MISMATCH:{split_name}"
    assert frame["source"].map(lambda value: value == "human").all(), f"NON_HUMAN_SOURCE:{split_name}"
    assert frame["selectedBySource"].map(
        lambda value: isinstance(value, (bool, np.bool_))
    ).all(), f"NON_BOOLEAN_SELECTED:{split_name}"

validate_candidate_source(train_source, "train")
validate_candidate_source(validation_source, "validation")
train_games = set(train_source["gameId"].tolist())
validation_games = set(validation_source["gameId"].tolist())

# Assertion
assert manifest["datasetSchemaVersion"] == 2
assert manifest["sourceFilter"] == "human"
assert train_games.isdisjoint(validation_games), "ORIGINAL_SPLIT_GAME_LEAKAGE"
assert len(train_source) == manifest["trainSampleCount"]
assert len(validation_source) == manifest["validationSampleCount"]
assert len(rejected_source) == manifest["rejectedPlacementLogCount"]

train_source = train_source.copy(deep=True)
validation_source = validation_source.copy(deep=True)
train_source["originalSplit"] = "train"
validation_source["originalSplit"] = "validation"
candidates = pd.concat([train_source, validation_source], ignore_index=True, copy=True)
assert len(candidates) == manifest["sampleCountBySource"]["human"]

# Output
display(pd.DataFrame([{
    "datasetSchemaVersion": manifest["datasetSchemaVersion"],
    "sourceFilter": manifest["sourceFilter"],
    "trainGames": len(train_games),
    "validationGames": len(validation_games),
    "gameIdLeakage": len(train_games.intersection(validation_games)),
    "combinedCandidates": len(candidates),
}]))

## 6. decisionKey와 placementKey 생성

decision은 gameId, playerId, pieceIndex로 식별하고 placement는 pieceType, rotation, row, col로 식별한다. 후보 수, 유일 label, 중복 placement와 board/piece/result context 일관성을 decision별로 검증한다.

In [ ]:
def is_integer(value):
    return isinstance(value, (int, np.integer)) and not isinstance(value, (bool, np.bool_))

def codepoint_key(value):
    return tuple(ord(character) for character in str(value))

def stable_json(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"))

def create_decision_key(row):
    game_id = row["gameId"]
    player_id = row["playerId"]
    piece_index = row["pieceIndex"]
    assert isinstance(game_id, str) and game_id
    assert isinstance(player_id, str) and player_id
    assert is_integer(piece_index)
    return f"{game_id}:{player_id}:{int(piece_index)}"

def create_placement_key(row):
    placement = row["candidatePlacement"]
    assert isinstance(placement, dict)
    values = [
        placement.get("pieceType"),
        placement.get("rotation"),
        placement.get("row"),
        placement.get("col"),
    ]
    assert isinstance(values[0], str) and values[0]
    assert all(is_integer(value) for value in values[1:])
    assert placement["pieceType"] == row["pieceType"]
    return f"{values[0]}:{int(values[1])}:{int(values[2])}:{int(values[3])}"

def decision_context_signature(row):
    metadata = row["metadata"]
    assert isinstance(metadata, dict)
    context = {
        "gameId": row["gameId"],
        "playerId": row["playerId"],
        "pieceIndex": int(row["pieceIndex"]),
        "source": row["source"],
        "pieceType": row["pieceType"],
        "aiDifficulty": row["aiDifficulty"],
        "boardBefore": row["boardBefore"],
        "metricsBefore": row["metricsBefore"],
        "isWinner": metadata.get("isWinner"),
        "winnerPlayerId": metadata.get("winnerPlayerId"),
        "finishType": metadata.get("finishType"),
    }
    return stable_json(context)

candidates["decisionKey"] = [create_decision_key(row) for _, row in candidates.iterrows()]
candidates["placementKey"] = [create_placement_key(row) for _, row in candidates.iterrows()]
candidates["contextSignature"] = [
    decision_context_signature(row) for _, row in candidates.iterrows()
]
decision_groups = candidates.groupby("decisionKey", sort=False, dropna=False)
decision_contract = decision_groups.agg(
    candidateCount=("placementKey", "size"),
    selectedCount=("selectedBySource", "sum"),
    placementCount=("placementKey", "nunique"),
    contextCount=("contextSignature", "nunique"),
    gameCount=("gameId", "nunique"),
)
decision_first = candidates.drop_duplicates("decisionKey", keep="first")
decision_difficulty_counts = {
    key: int(value)
    for key, value in decision_first["aiDifficulty"].value_counts().to_dict().items()
}
candidate_difficulty_counts = {
    key: int(value)
    for key, value in candidates["aiDifficulty"].value_counts().to_dict().items()
}

# Assertion
assert (decision_contract["candidateCount"] >= 2).all()
assert (decision_contract["selectedCount"] == 1).all()
assert (
    decision_contract["placementCount"] == decision_contract["candidateCount"]
).all(), "DUPLICATE_PLACEMENT"
assert (decision_contract["contextCount"] == 1).all(), "INCONSISTENT_DECISION_CONTEXT"
assert (decision_contract["gameCount"] == 1).all()
assert len(decision_contract) == manifest["acceptedPlacementLogCount"]
assert candidates["selectedBySource"].sum() == manifest["selectedBySourceSampleCount"]
assert candidates["gameId"].nunique() == manifest["totalGameCount"]
assert len(train_games) == manifest["trainGameCount"]
assert len(validation_games) == manifest["validationGameCount"]
assert manifest["acceptedPlacementLogCountBySource"] == {
    "human": len(decision_contract),
    "heuristic_ai": 0,
}
assert manifest["sampleCountBySource"] == {
    "human": len(candidates),
    "heuristic_ai": 0,
}
assert manifest["acceptedPlacementLogCountByAiDifficulty"] == {
    "normal": decision_difficulty_counts.get("normal", 0),
    "hard": decision_difficulty_counts.get("hard", 0),
    "unknown": decision_difficulty_counts.get("unknown", 0),
}
assert manifest["sampleCountByAiDifficulty"] == {
    "normal": candidate_difficulty_counts.get("normal", 0),
    "hard": candidate_difficulty_counts.get("hard", 0),
    "unknown": candidate_difficulty_counts.get("unknown", 0),
}

# Output
display(pd.DataFrame([{
    "games": candidates["gameId"].nunique(),
    "decisions": len(decision_contract),
    "candidates": len(candidates),
    "minimumCandidatesPerDecision": int(decision_contract["candidateCount"].min()),
    "maximumCandidatesPerDecision": int(decision_contract["candidateCount"].max()),
    "invalidDecisionGroups": 0,
}]))

## 7. 명시적인 feature allowlist

feature schema v1의 이름과 순서를 코드로 고정한다. label, outcome metadata, heuristic score, input sequence와 식별자는 feature allowlist에 포함되지 않는다.

In [ ]:
FEATURE_SCHEMA_VERSION = 1
FEATURE_NAMES = [
    "candidateResult.linesCleared",
    "candidateResult.gameOver",
    "metricDelta.aggregateHeight",
    "metricDelta.maxHeight",
    "metricDelta.holes",
    "metricDelta.bumpiness",
    "metricDelta.columnHeights[0]",
    "metricDelta.columnHeights[1]",
    "metricDelta.columnHeights[2]",
    "metricDelta.columnHeights[3]",
    "metricDelta.columnHeights[4]",
    "metricDelta.columnHeights[5]",
    "metricDelta.columnHeights[6]",
    "metricDelta.columnHeights[7]",
    "metricDelta.columnHeights[8]",
    "metricDelta.columnHeights[9]",
]
FORBIDDEN_FEATURES = {
    "selectedBySource",
    "metadata.isWinner",
    "winnerPlayerId",
    "heuristicBaseline.score",
    "metadata.inputSequence",
    "gameId",
    "playerId",
}

# Assertion
assert len(FEATURE_NAMES) == 16
assert len(set(FEATURE_NAMES)) == 16
assert set(FEATURE_NAMES).isdisjoint(FORBIDDEN_FEATURES)

# Output
display(pd.DataFrame({
    "featureIndex": range(len(FEATURE_NAMES)),
    "featureName": FEATURE_NAMES,
}))

## 8. feature DataFrame 생성

각 nested candidate row에서 allowlist 16개만 순서대로 꺼낸다. null, NaN, Infinity, 잘못된 boolean이나 column height 길이를 거부하고 gameId, decisionKey, placementKey Unicode code-point 순서로 명시적으로 정렬한다.

In [ ]:
def finite_number(value):
    return (
        isinstance(value, (int, float, np.integer, np.floating))
        and not isinstance(value, (bool, np.bool_))
        and math.isfinite(float(value))
    )

def extract_feature_vector(row):
    result = row["candidateResult"]
    delta = row["metricDelta"]
    assert isinstance(result, dict)
    assert isinstance(delta, dict)
    lines_cleared = result.get("linesCleared")
    game_over = result.get("gameOver")
    assert is_integer(lines_cleared) and int(lines_cleared) >= 0
    assert isinstance(game_over, (bool, np.bool_))
    scalar_names = ["aggregateHeight", "maxHeight", "holes", "bumpiness"]
    scalar_values = [delta.get(name) for name in scalar_names]
    assert all(finite_number(value) for value in scalar_values)
    column_heights = delta.get("columnHeights")
    assert isinstance(column_heights, list) and len(column_heights) == 10
    assert all(finite_number(value) for value in column_heights)
    vector = np.asarray(
        [float(lines_cleared), float(bool(game_over))]
        + [float(value) for value in scalar_values]
        + [float(value) for value in column_heights],
        dtype=np.float64,
    )
    assert vector.shape == (len(FEATURE_NAMES),)
    assert np.isfinite(vector).all()
    return vector

feature_matrix = np.vstack([
    extract_feature_vector(row) for _, row in candidates.iterrows()
])
feature_frame = candidates[
    [
        "gameId",
        "playerId",
        "pieceIndex",
        "decisionKey",
        "placementKey",
        "selectedBySource",
        "aiDifficulty",
        "metadata",
        "originalSplit",
    ]
].copy(deep=True)
for feature_index, feature_name in enumerate(FEATURE_NAMES):
    feature_frame[feature_name] = feature_matrix[:, feature_index]
feature_frame["isWinner"] = feature_frame["metadata"].map(
    lambda value: value.get("isWinner") if isinstance(value, dict) else None
)
canonical_indices = sorted(
    feature_frame.index.tolist(),
    key=lambda index: (
        codepoint_key(feature_frame.at[index, "gameId"]),
        codepoint_key(feature_frame.at[index, "decisionKey"]),
        codepoint_key(feature_frame.at[index, "placementKey"]),
    ),
)
feature_frame = feature_frame.loc[canonical_indices].reset_index(drop=True)
feature_frame["candidateIndex"] = np.arange(len(feature_frame), dtype=np.int64)
canonical_keys = list(zip(
    feature_frame["gameId"],
    feature_frame["decisionKey"],
    feature_frame["placementKey"],
))

# Assertion
assert canonical_keys == sorted(
    canonical_keys,
    key=lambda values: tuple(codepoint_key(value) for value in values),
)
assert np.isfinite(
    feature_frame[FEATURE_NAMES].to_numpy(dtype=np.float64, copy=True)
).all()
assert feature_frame["selectedBySource"].map(
    lambda value: isinstance(value, (bool, np.bool_))
).all()

# Output
display(feature_frame[
    ["gameId", "decisionKey", "placementKey", "selectedBySource"] + FEATURE_NAMES[:4]
].head(5))

## 9. 데이터와 label 분포

모델을 보기 전에 game, 난이도, 승패 segmentation과 candidate 수 분포를 작은 집계표로 확인한다. isWinner는 분석 segmentation이며 feature matrix에는 들어가지 않는다.

In [ ]:
decision_view = feature_frame.drop_duplicates("decisionKey", keep="first")
candidate_count_distribution = (
    feature_frame.groupby("decisionKey", sort=False)
    .size()
    .describe(percentiles=[0.25, 0.5, 0.75])
    .to_frame(name="candidateCount")
)
difficulty_distribution = (
    feature_frame.groupby("aiDifficulty", sort=False)
    .agg(candidates=("placementKey", "size"), decisions=("decisionKey", "nunique"))
    .reset_index()
)
winner_distribution = (
    decision_view.groupby("isWinner", sort=False)
    .agg(decisions=("decisionKey", "size"), games=("gameId", "nunique"))
    .reset_index()
)

# Assertion
assert "isWinner" not in FEATURE_NAMES
assert int(decision_view["decisionKey"].nunique()) == len(decision_contract)
assert feature_frame["selectedBySource"].value_counts().to_dict()[True] == len(decision_contract)

# Output
display(difficulty_distribution)
display(winner_distribution)
display(candidate_count_distribution)

## 10. duplicate/zero-difference/tie 진단

selected candidate와 동일한 raw feature vector를 가진 unselected candidate는 어떤 선형 모델도 구분할 수 없다. 이를 selected-feature duplicate와 zero-difference pair로 분리 집계한다. 모델 score tie는 fold 평가 후 추가한다.

In [ ]:
representation_rows = []
for decision_key in sorted(feature_frame["decisionKey"].unique(), key=codepoint_key):
    group = feature_frame.loc[feature_frame["decisionKey"] == decision_key]
    group_features = group[FEATURE_NAMES].to_numpy(dtype=np.float64, copy=True)
    selected_mask = group["selectedBySource"].to_numpy(dtype=bool, copy=True)
    assert selected_mask.sum() == 1
    selected_features = group_features[selected_mask][0]
    unselected_features = group_features[~selected_mask]
    zero_difference_mask = np.all(
        unselected_features == selected_features,
        axis=1,
    )
    representation_rows.append({
        "decisionKey": decision_key,
        "zeroDifferencePairCount": int(zero_difference_mask.sum()),
        "selectedFeatureDuplicate": bool(zero_difference_mask.any()),
    })
representation_frame = pd.DataFrame(representation_rows)
representation_by_decision = representation_frame.set_index("decisionKey").to_dict("index")
REPRESENTATION_SUMMARY = {
    "pairCount": int(len(feature_frame) - feature_frame["decisionKey"].nunique()),
    "zeroDifferencePairCount": int(representation_frame["zeroDifferencePairCount"].sum()),
    "zeroDifferenceDecisionCount": int(
        (representation_frame["zeroDifferencePairCount"] > 0).sum()
    ),
    "selectedFeatureDuplicateDecisionCount": int(
        representation_frame["selectedFeatureDuplicate"].sum()
    ),
}

# Assertion
assert REPRESENTATION_SUMMARY["pairCount"] > 0
assert (
    REPRESENTATION_SUMMARY["zeroDifferenceDecisionCount"]
    == REPRESENTATION_SUMMARY["selectedFeatureDuplicateDecisionCount"]
)

# Output
display(pd.DataFrame([REPRESENTATION_SUMMARY]))

## 11. gameId 기준 LOGO split

원본 train/validation 표시는 LOGO 입력에서 제거하고, manifest의 모든 gameId를 각각 정확히 한 번 validation으로 사용한다.

In [ ]:
GAME_IDS = sorted(feature_frame["gameId"].unique().tolist(), key=codepoint_key)
fold_plan_rows = []
for validation_game_id in GAME_IDS:
    validation_mask = feature_frame["gameId"] == validation_game_id
    fold_plan_rows.append({
        "validationGameId": validation_game_id,
        "trainGames": int(feature_frame.loc[~validation_mask, "gameId"].nunique()),
        "validationGames": int(feature_frame.loc[validation_mask, "gameId"].nunique()),
        "trainDecisions": int(feature_frame.loc[~validation_mask, "decisionKey"].nunique()),
        "validationDecisions": int(feature_frame.loc[validation_mask, "decisionKey"].nunique()),
        "trainCandidates": int((~validation_mask).sum()),
        "validationCandidates": int(validation_mask.sum()),
    })
fold_plan = pd.DataFrame(fold_plan_rows)

# Assertion
assert len(GAME_IDS) == manifest["totalGameCount"]
assert fold_plan["validationGames"].eq(1).all()
assert fold_plan["trainGames"].eq(len(GAME_IDS) - 1).all()
assert fold_plan["validationDecisions"].sum() == len(decision_contract)

# Output
display(fold_plan)

## 12. fold별 train-only StandardScaler

각 fold scaler는 train candidate matrix에 한 번만 fit한다. validation은 transform만 하며 zero-variance feature의 scale이 1인지 확인한다.

In [ ]:
def fit_train_only_scaler(train_frame, validation_frame):
    train_values = train_frame[FEATURE_NAMES].to_numpy(dtype=np.float64, copy=True)
    validation_values = validation_frame[FEATURE_NAMES].to_numpy(dtype=np.float64, copy=True)
    scaler = StandardScaler(with_mean=True, with_std=True)
    train_scaled = scaler.fit_transform(train_values)
    validation_scaled = scaler.transform(validation_values)
    zero_variance_mask = np.var(train_values, axis=0) == 0.0

    # Assertion
    assert int(scaler.n_samples_seen_) == len(train_frame)
    assert scaler.with_mean is True and scaler.with_std is True
    assert np.isfinite(train_scaled).all() and np.isfinite(validation_scaled).all()
    assert np.all(scaler.scale_[zero_variance_mask] == 1.0)

    return scaler, train_scaled, validation_scaled, int(zero_variance_mask.sum())

# Output
{"scaler": "StandardScaler", "fitScope": "fold train candidates only"}

## 13. decision-balanced pair 생성

train decision마다 selected candidate와 모든 unselected candidate의 standardized feature 차이를 만든다. 각 decision의 원래 pair weight 합은 정확히 1이다.

In [ ]:
WEIGHT_TOLERANCE = 1e-12

def create_decision_balanced_pairs(train_frame, train_scaled):
    scaled_by_index = {
        frame_index: train_scaled[position]
        for position, frame_index in enumerate(train_frame.index.tolist())
    }
    pair_rows = []
    decision_keys = sorted(train_frame["decisionKey"].unique().tolist(), key=codepoint_key)
    for decision_key in decision_keys:
        group = train_frame.loc[train_frame["decisionKey"] == decision_key]
        selected = group.loc[group["selectedBySource"]]
        unselected = group.loc[~group["selectedBySource"]]
        assert len(selected) == 1 and not unselected.empty
        selected_index = selected.index[0]
        pair_weight = 1.0 / len(unselected)
        for unselected_index in unselected.index.tolist():
            difference = (
                scaled_by_index[selected_index] - scaled_by_index[unselected_index]
            )
            pair_rows.append({
                "decisionKey": decision_key,
                "selectedIndex": int(selected_index),
                "unselectedIndex": int(unselected_index),
                "difference": difference.astype(np.float64, copy=True),
                "weight": float(pair_weight),
            })
    pairs = pd.DataFrame(pair_rows)
    decision_weight_sums = pairs.groupby("decisionKey", sort=False)["weight"].sum()

    # Assertion
    assert len(decision_weight_sums) == len(decision_keys)
    assert np.allclose(
        decision_weight_sums.to_numpy(dtype=np.float64),
        np.ones(len(decision_weight_sums)),
        rtol=0.0,
        atol=WEIGHT_TOLERANCE,
    )
    return pairs

# Output
{"pairDefinition": "selected standardized feature - unselected standardized feature"}

## 14. symmetric binary sample 생성

각 pair를 positive difference와 negative difference 두 sample로 바꾸고 pair weight를 절반씩 나눈다. 실제 sklearn에 전달할 sample_weight 배열에서 양쪽 weight와 decision별 합을 다시 검증한다.

In [ ]:
def create_symmetric_samples(pairs):
    differences = np.vstack(pairs["difference"].to_numpy())
    pair_weights = pairs["weight"].to_numpy(dtype=np.float64, copy=True)
    pair_count, feature_count = differences.shape
    x_values = np.empty((pair_count * 2, feature_count), dtype=np.float64)
    labels = np.empty(pair_count * 2, dtype=np.int64)
    sample_weights = np.empty(pair_count * 2, dtype=np.float64)
    x_values[0::2] = differences
    x_values[1::2] = -differences
    labels[0::2] = 1
    labels[1::2] = 0
    sample_weights[0::2] = pair_weights / 2.0
    sample_weights[1::2] = pair_weights / 2.0
    sample_decision_keys = np.repeat(pairs["decisionKey"].to_numpy(), 2)

    # Assertion
    assert np.array_equal(labels[0::2], np.ones(pair_count, dtype=np.int64))
    assert np.array_equal(labels[1::2], np.zeros(pair_count, dtype=np.int64))
    assert np.allclose(sample_weights[0::2], pair_weights / 2.0, rtol=0.0, atol=0.0)
    assert np.allclose(sample_weights[1::2], pair_weights / 2.0, rtol=0.0, atol=0.0)
    transformed_weights = pd.DataFrame({
        "decisionKey": sample_decision_keys,
        "sampleWeight": sample_weights,
    }).groupby("decisionKey", sort=False)["sampleWeight"].sum()
    assert np.allclose(
        transformed_weights.to_numpy(dtype=np.float64),
        np.ones(len(transformed_weights)),
        rtol=0.0,
        atol=WEIGHT_TOLERANCE,
    )
    assert np.isfinite(x_values).all() and np.isfinite(sample_weights).all()

    diagnostics = {
        "minimumDecisionSampleWeight": float(transformed_weights.min()),
        "maximumDecisionSampleWeight": float(transformed_weights.max()),
    }
    return x_values, labels, sample_weights, diagnostics

# Output
{"symmetricSamplesPerPair": 2, "decisionSampleWeightSum": 1.0}

## 15. sklearn LogisticRegression 학습

하이퍼파라미터 탐색 없이 검증된 baseline 하나만 사용한다. C는 fold의 train decision 수와 l2=0.01에서 계산하며 convergence warning은 hard failure다.

In [ ]:
MODEL_CONFIG = {
    "model": "LogisticRegression",
    "solver": "lbfgs",
    "fitIntercept": False,
    "classWeight": None,
    "tolerance": 1e-12,
    "maxIterations": 10000,
    "l2": 0.01,
    "cFormula": "1/(trainDecisionCount*l2)",
}

def fit_pairwise_model(x_values, labels, sample_weights, train_decision_count):
    c_value = 1.0 / (train_decision_count * MODEL_CONFIG["l2"])
    model = LogisticRegression(
        C=c_value,
        solver=MODEL_CONFIG["solver"],
        fit_intercept=MODEL_CONFIG["fitIntercept"],
        class_weight=MODEL_CONFIG["classWeight"],
        tol=MODEL_CONFIG["tolerance"],
        max_iter=MODEL_CONFIG["maxIterations"],
    )
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        model.fit(x_values, labels, sample_weight=sample_weights)
    convergence_warnings = [
        str(item.message)
        for item in caught
        if issubclass(item.category, ConvergenceWarning)
    ]
    unexpected_warnings = [
        f"{item.category.__name__}:{item.message}"
        for item in caught
        if not issubclass(item.category, ConvergenceWarning)
    ]
    iteration_count = int(model.n_iter_[0])

    # Assertion
    assert not convergence_warnings, convergence_warnings
    assert iteration_count < MODEL_CONFIG["maxIterations"]
    assert model.coef_.shape == (1, len(FEATURE_NAMES))
    assert np.isfinite(model.coef_).all()

    return model, c_value, iteration_count, convergence_warnings, unexpected_warnings

# Output
display(pd.DataFrame([MODEL_CONFIG]))

## 16. fold별 metric

validation candidate score는 해당 fold의 train-only scaler와 sklearn coefficient로 계산한다. 동점은 score 내림차순 뒤 placementKey Unicode code-point 순서로 해결한다. tie-aware Top-1은 selected가 최고 score tie set에 포함되는지 보는 낙관적 표현 진단이다.

In [ ]:
def summarize_predictions(predictions):
    prediction_frame = pd.DataFrame(predictions)
    ranks = prediction_frame["selectedRank"].to_numpy(dtype=np.int64)
    count = len(prediction_frame)
    assert count > 0
    return {
        "decisionCount": int(count),
        "top1Accuracy": float(np.mean(ranks == 1)),
        "top3Accuracy": float(np.mean(ranks <= 3)),
        "mrr": float(np.mean(1.0 / ranks)),
        "meanSelectedRank": float(np.mean(ranks)),
        "tieAwareTop1Accuracy": float(
            prediction_frame["selectedAtTopScore"].mean()
        ),
        "topScoreTieDecisionCount": int(
            (prediction_frame["topScoreTieCount"] > 1).sum()
        ),
        "selectedFeatureDuplicateDecisionCount": int(
            prediction_frame["selectedFeatureDuplicate"].sum()
        ),
    }

def score_validation_candidates(validation_frame, validation_scaled, weights):
    score_by_index = {
        frame_index: float(validation_scaled[position] @ weights)
        for position, frame_index in enumerate(validation_frame.index.tolist())
    }
    assert all(math.isfinite(score) for score in score_by_index.values())
    predictions = []
    decision_keys = sorted(
        validation_frame["decisionKey"].unique().tolist(),
        key=codepoint_key,
    )
    for decision_key in decision_keys:
        group = validation_frame.loc[
            validation_frame["decisionKey"] == decision_key
        ]
        candidate_rows = []
        for frame_index, row in group.iterrows():
            candidate_rows.append({
                "placementKey": row["placementKey"],
                "selected": bool(row["selectedBySource"]),
                "score": score_by_index[frame_index],
            })
        candidate_rows.sort(key=lambda row: codepoint_key(row["placementKey"]))
        ranked = sorted(
            candidate_rows,
            key=lambda row: (-row["score"], codepoint_key(row["placementKey"])),
        )
        selected = next(row for row in candidate_rows if row["selected"])
        selected_rank = next(
            index + 1
            for index, row in enumerate(ranked)
            if row["placementKey"] == selected["placementKey"]
        )
        top_score = ranked[0]["score"]
        top_score_tie_count = sum(
            row["score"] == top_score for row in candidate_rows
        )
        predictions.append({
            "gameId": str(group["gameId"].iloc[0]),
            "decisionKey": decision_key,
            "aiDifficulty": str(group["aiDifficulty"].iloc[0]),
            "isWinner": bool(group["isWinner"].iloc[0]),
            "selectedRank": int(selected_rank),
            "selectedAtTopScore": bool(selected["score"] == top_score),
            "topScoreTieCount": int(top_score_tie_count),
            "selectedFeatureDuplicate": bool(
                representation_by_decision[decision_key]["selectedFeatureDuplicate"]
            ),
        })
    return predictions

fold_results = []
held_out_predictions = []
validation_game_occurrences = []
for validation_game_id in GAME_IDS:
    validation_mask = feature_frame["gameId"] == validation_game_id
    train_fold = feature_frame.loc[~validation_mask].copy()
    validation_fold = feature_frame.loc[validation_mask].copy()
    train_game_ids = set(train_fold["gameId"].tolist())
    validation_game_ids = set(validation_fold["gameId"].tolist())
    assert train_game_ids.isdisjoint(validation_game_ids)

    scaler, train_scaled, validation_scaled, zero_variance_count = (
        fit_train_only_scaler(train_fold, validation_fold)
    )
    pairs = create_decision_balanced_pairs(train_fold, train_scaled)
    x_values, labels, sample_weights, sample_weight_diagnostics = (
        create_symmetric_samples(pairs)
    )
    train_decision_count = int(train_fold["decisionKey"].nunique())
    model, c_value, iteration_count, convergence_warnings, unexpected_warnings = (
        fit_pairwise_model(
            x_values,
            labels,
            sample_weights,
            train_decision_count,
        )
    )
    fold_predictions = score_validation_candidates(
        validation_fold,
        validation_scaled,
        model.coef_[0].astype(np.float64, copy=True),
    )
    fold_metrics = summarize_predictions(fold_predictions)
    zero_pair_mask = np.vstack(pairs["difference"].to_numpy()) == 0.0
    zero_pair_mask = zero_pair_mask.all(axis=1)
    zero_difference_decisions = pairs.loc[
        zero_pair_mask, "decisionKey"
    ].nunique()

    fold_results.append({
        "validationGameId": validation_game_id,
        "trainGameCount": int(len(train_game_ids)),
        "validationGameCount": int(len(validation_game_ids)),
        "gameIdLeakageCount": int(len(train_game_ids.intersection(validation_game_ids))),
        "trainDecisionCount": train_decision_count,
        "validationDecisionCount": int(validation_fold["decisionKey"].nunique()),
        "trainCandidateCount": int(len(train_fold)),
        "validationCandidateCount": int(len(validation_fold)),
        "scalerFitCandidateCount": int(scaler.n_samples_seen_),
        "scalerFitScope": "train_candidates_only",
        "zeroVarianceFeatureCount": zero_variance_count,
        "trainPairCount": int(len(pairs)),
        "zeroDifferencePairCount": int(zero_pair_mask.sum()),
        "zeroDifferenceDecisionCount": int(zero_difference_decisions),
        "minimumDecisionSampleWeight": sample_weight_diagnostics[
            "minimumDecisionSampleWeight"
        ],
        "maximumDecisionSampleWeight": sample_weight_diagnostics[
            "maximumDecisionSampleWeight"
        ],
        "c": float(c_value),
        "iterationCount": iteration_count,
        "convergenceWarnings": convergence_warnings,
        "unexpectedWarnings": unexpected_warnings,
        "converged": True,
        "metrics": fold_metrics,
    })
    held_out_predictions.extend(fold_predictions)
    validation_game_occurrences.append(validation_game_id)

# Assertion
assert validation_game_occurrences == GAME_IDS
assert len(set(validation_game_occurrences)) == len(GAME_IDS)
assert sum(result["validationDecisionCount"] for result in fold_results) == len(decision_contract)
assert all(result["gameIdLeakageCount"] == 0 for result in fold_results)
assert all(
    result["scalerFitCandidateCount"] == result["trainCandidateCount"]
    for result in fold_results
)
assert all(result["converged"] for result in fold_results)

# Output
fold_display = pd.DataFrame([{
    "validationGameId": result["validationGameId"],
    "decisions": result["validationDecisionCount"],
    "pairs": result["trainPairCount"],
    "iterations": result["iterationCount"],
    "top1": result["metrics"]["top1Accuracy"],
    "top3": result["metrics"]["top3Accuracy"],
    "mrr": result["metrics"]["mrr"],
    "tieAwareTop1": result["metrics"]["tieAwareTop1Accuracy"],
    "topScoreTies": result["metrics"]["topScoreTieDecisionCount"],
} for result in fold_results])
display(fold_display)

## 17. micro/macro/median/range/worst-fold 집계

micro는 모든 fold의 held-out prediction을 직접 합쳐 계산한다. macro, median, range는 fold metric의 분포를 보여준다. worst fold는 canonical Top-1이 가장 낮고, 동률이면 MRR이 낮은 fold로 정한다.

In [ ]:
AGGREGATED_METRIC_NAMES = [
    "top1Accuracy",
    "top3Accuracy",
    "mrr",
    "meanSelectedRank",
    "tieAwareTop1Accuracy",
]
micro_metrics = summarize_predictions(held_out_predictions)

def summarize_segments(predictions, field_name, value_names):
    prediction_frame = pd.DataFrame(predictions)
    segments = {}
    for value, segment_name in value_names:
        segment_records = prediction_frame.loc[
            prediction_frame[field_name] == value
        ].to_dict(orient="records")
        if not segment_records:
            continue
        metrics = summarize_predictions(segment_records)
        metrics["gameCount"] = int(len({
            record["gameId"] for record in segment_records
        }))
        segments[segment_name] = metrics
    return segments

segment_metrics = {
    "aiDifficulty": summarize_segments(
        held_out_predictions,
        "aiDifficulty",
        [("normal", "normal"), ("hard", "hard")],
    ),
    "outcome": summarize_segments(
        held_out_predictions,
        "isWinner",
        [(True, "winner"), (False, "loss")],
    ),
}
fold_metric_frame = pd.DataFrame([
    {
        "validationGameId": result["validationGameId"],
        "decisionCount": result["metrics"]["decisionCount"],
        **{
            name: result["metrics"][name]
            for name in AGGREGATED_METRIC_NAMES
        },
    }
    for result in fold_results
])
macro_average = {
    name: float(fold_metric_frame[name].mean())
    for name in AGGREGATED_METRIC_NAMES
}
median = {
    name: float(fold_metric_frame[name].median())
    for name in AGGREGATED_METRIC_NAMES
}
metric_range = {
    name: {
        "minimum": float(fold_metric_frame[name].min()),
        "maximum": float(fold_metric_frame[name].max()),
    }
    for name in AGGREGATED_METRIC_NAMES
}
worst_fold = min(
    fold_results,
    key=lambda result: (
        result["metrics"]["top1Accuracy"],
        result["metrics"]["mrr"],
        codepoint_key(result["validationGameId"]),
    ),
)
aggregate = {
    "foldCount": int(len(fold_results)),
    "convergedFoldCount": int(sum(result["converged"] for result in fold_results)),
    "validationGameCount": int(len(validation_game_occurrences)),
    "validationGameOccurrenceCount": int(len(validation_game_occurrences)),
    "gameIdLeakageCount": int(sum(
        result["gameIdLeakageCount"] for result in fold_results
    )),
    "micro": micro_metrics,
    "macroAverage": macro_average,
    "median": median,
    "range": metric_range,
    "segments": segment_metrics,
    "worstFold": {
        "validationGameId": worst_fold["validationGameId"],
        "decisionCount": worst_fold["validationDecisionCount"],
        "top1Accuracy": worst_fold["metrics"]["top1Accuracy"],
        "mrr": worst_fold["metrics"]["mrr"],
    },
}

# Assertion
assert aggregate["foldCount"] == aggregate["convergedFoldCount"] == len(GAME_IDS)
assert aggregate["gameIdLeakageCount"] == 0
assert micro_metrics["decisionCount"] == manifest["acceptedPlacementLogCount"]
assert len({row["decisionKey"] for row in held_out_predictions}) == micro_metrics["decisionCount"]

# Output
display(pd.DataFrame([{
    **micro_metrics,
    "macroTop1": macro_average["top1Accuracy"],
    "medianTop1": median["top1Accuracy"],
    "minimumTop1": metric_range["top1Accuracy"]["minimum"],
    "maximumTop1": metric_range["top1Accuracy"]["maximum"],
    "worstFold": aggregate["worstFold"]["validationGameId"],
}]))
display(pd.DataFrame([
    {"segmentType": segment_type, "segment": segment_name, **metrics}
    for segment_type, segments in segment_metrics.items()
    for segment_name, metrics in segments.items()
]))

## 18. 결정성 및 원본 불변성 확인

학습 후 네 source checksum을 다시 계산한다. report에는 timestamp, 사용자 경로, 실행별 임시 이름을 넣지 않고 같은 디렉터리의 무작위 exclusive temp file을 연 handle로 기록한 뒤 atomic replace한다.

In [ ]:
SOURCE_SHA256_AFTER = {name: sha256_file(path) for name, path in SOURCE_FILES.items()}

# Assertion
assert SOURCE_SHA256_AFTER == SOURCE_SHA256_BEFORE, "SOURCE_DATASET_MUTATED"

EXPERIMENT_STATUS = "continue_experimenting"
report = {
    "experimentReportSchemaVersion": 1,
    "experimentName": EXPERIMENT_NAME,
    "experimentStatus": EXPERIMENT_STATUS,
    "humanChoiceImitationOnly": True,
    "runtimeIntegration": False,
    "productionModelUpdated": False,
    "sourceDataset": {
        "relativeDirectory": DATASET_RELATIVE_DIR.as_posix(),
        "datasetSchemaVersion": int(manifest["datasetSchemaVersion"]),
        "sourceFilter": manifest["sourceFilter"],
        "sourceFileSha256": SOURCE_SHA256_BEFORE,
        "sourceUnchanged": True,
    },
    "environment": ACTUAL_ENVIRONMENT,
    "featureContract": {
        "featureSchemaVersion": FEATURE_SCHEMA_VERSION,
        "featureNames": FEATURE_NAMES,
        "selectedBySourceUsage": "label_only",
        "outcomeMetadataUsage": "evaluation_segmentation_only",
    },
    "modelConfig": MODEL_CONFIG,
    "dataSummary": {
        "gameCount": int(feature_frame["gameId"].nunique()),
        "decisionCount": int(feature_frame["decisionKey"].nunique()),
        "candidateCount": int(len(feature_frame)),
        "selectedCandidateCount": int(feature_frame["selectedBySource"].sum()),
        "originalTrainGameCount": int(len(train_games)),
        "originalValidationGameCount": int(len(validation_games)),
        "originalSplitGameIdLeakageCount": int(len(train_games.intersection(validation_games))),
    },
    "fullDatasetRepresentation": REPRESENTATION_SUMMARY,
    "folds": fold_results,
    "aggregate": aggregate,
    "interpretation": {
        "canonicalRankingRemainsPrimary": True,
        "tieAwareTop1IsOptimisticRepresentationDiagnostic": True,
        "gameplayStrengthClaimed": False,
        "nextDecision": "Keep experimenting because feature duplicates and fold variance remain, and the user has not approved freezing.",
    },
}

def atomic_write_report(output_path, text, protected_sources):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    assert_output_separate_from_sources(output_path, protected_sources)
    descriptor, staging_name = tempfile.mkstemp(
        prefix=f".{output_path.name}.",
        suffix=".tmp",
        dir=output_path.parent,
    )
    staging_path = Path(staging_name)
    descriptor_stat = os.fstat(descriptor)
    try:
        with os.fdopen(descriptor, "w", encoding="utf-8", newline="\n") as handle:
            handle.write(text)
            handle.flush()
            os.fsync(handle.fileno())
        staging_stat = staging_path.lstat()
        assert not staging_path.is_symlink(), "REPORT_STAGING_SYMLINK_NOT_ALLOWED"
        assert (
            staging_stat.st_dev,
            staging_stat.st_ino,
        ) == (
            descriptor_stat.st_dev,
            descriptor_stat.st_ino,
        ), "REPORT_STAGING_IDENTITY_CHANGED"
        for source_path in protected_sources:
            assert not same_inode(staging_path, source_path), "REPORT_STAGING_SOURCE_COLLISION"
        assert_output_separate_from_sources(output_path, protected_sources)
        os.replace(staging_path, output_path)
    finally:
        if staging_path.exists():
            staging_path.unlink()

report_text = json.dumps(
    report,
    ensure_ascii=False,
    indent=2,
    sort_keys=True,
    allow_nan=False,
) + "\n"
atomic_write_report(REPORT_PATH, report_text, list(SOURCE_FILES.values()))
REPORT_SHA256 = hashlib.sha256(report_text.encode("utf-8")).hexdigest()

# Assertion
assert sha256_file(REPORT_PATH) == REPORT_SHA256
assert "timestamp" not in report_text.lower()
assert str(ROOT) not in report_text

# Output
display(pd.DataFrame([{
    "reportSha256": REPORT_SHA256,
    "sourceUnchanged": SOURCE_SHA256_AFTER == SOURCE_SHA256_BEFORE,
    "experimentStatus": EXPERIMENT_STATUS,
    "productionModelUpdated": report["productionModelUpdated"],
}]))

## 19. 결과 해석과 다음 실험 판단

실험 상태: **continue_experimenting**

현재 baseline은 leakage 없이 manifest game 수와 같은 fold의 held-out 평가를 만들고 재현 가능 report를 생성한다. 그러나 selected-feature duplicate와 zero-difference pair가 남아 있고 fold 편차 및 worst fold를 더 검토해야 한다. 또한 사용자가 feature/model 동결을 승인하지 않았다.

따라서 이번 작업은 최종 모델을 학습하거나 production/PVE에 연결하지 않는다. 다음 실험에서는 label leakage 없이 후보 구분력을 높일 수 있는 feature 가설을 한 번에 하나씩 검증한다.